# 12 · Pilot on the new (jittered) crops: do the models beat the no-network baseline?
**Runtime: T4 GPU.** Run all. About 3–4 GPU hours. Resumable: after a disconnect just **Run all** again; finished runs are skipped.

Data: `data/processed/cbis_mass_256_jit` (notebook 11). Same code, model library (`qseg2`), folds and settings as the main runs.
Fold 0, seed 0 only. This is a feasibility check, **not** part of the final results: the final runs go to a new folder and
start from scratch, so nothing is selected from the pilot.

* Part A (main schedule, as in the paper): `unet`, `g_q`, `g_cl`, `p_q`, `p_cl` → `results/p2_jit_pilot`
* Part B (learning-rate schedule on the training loss): `p_q`, `p_cl` → `results/p2_jit_pilot_lrtrain`

**Pass rule (fixed before the runs):** on fold 0 the U-Net and at least the global models must beat the centred-disc baseline
of the same fold by 0.10 Dice or more. Cell 6 prints the comparison.


In [ ]:
# 1 · Drive, paths
from google.colab import drive
drive.mount('/content/drive')
import os, sys, glob, importlib

PROJECT  = '/content/drive/MyDrive/quantum-placement'
DATA_DIR = f'{PROJECT}/data/processed/cbis_mass_256_jit'
CODE_DIR = f'{PROJECT}/code'
RUNS_DIR = f'{PROJECT}/results/p2_jit_pilot'             # part A (main schedule)
LR_DIR   = f'{PROJECT}/results/p2_jit_pilot_lrtrain'     # part B (schedule on training loss)
TIME_DIR = f'{PROJECT}/results/_timing'
QSEG_DIR = '/content/drive/MyDrive/quantum-placement/code'            # folder that contains the qseg/ package
if not os.path.exists(f'{QSEG_DIR}/qseg/quantum_sim.py'):
    hits = glob.glob('/content/drive/MyDrive/**/qseg/quantum_sim.py', recursive=True)
    assert hits, 'qseg package not found on Drive'
    QSEG_DIR = os.path.dirname(os.path.dirname(hits[0]))
for d in (CODE_DIR, RUNS_DIR, LR_DIR, TIME_DIR): os.makedirs(d, exist_ok=True)
assert os.path.exists(f'{DATA_DIR}/folds.csv'), 'Run notebook 11 first'
for p in (QSEG_DIR, CODE_DIR):
    if p not in sys.path: sys.path.insert(0, p)
print('qseg :', QSEG_DIR); print('data :', DATA_DIR); print('runs :', RUNS_DIR)


In [ ]:
# 2 · Use the model library already on Drive (written by notebook 02; not rewritten here)
import importlib, qseg2; importlib.reload(qseg2)
import tensorflow as tf, numpy as np, pandas as pd
print('qseg2 from:', qseg2.__file__)
print('TF', tf.__version__, '| GPU:', tf.config.list_physical_devices('GPU'))
assert tf.config.list_physical_devices('GPU'), 'No GPU: Runtime -> Change runtime type -> T4 GPU'


In [ ]:
# 3 · Data + shared config (base settings of the qseg package)
from qseg.config import Config
cfg = Config(work_dir=RUNS_DIR, max_epochs=200)   # v3: up to 200 epochs; batch 16, Adam 1e-4, early stop patience 10 after epoch 15
X, Y, DF = qseg2.load_cbis(DATA_DIR)
print(X.shape, Y.shape, 'ROIs:', len(DF), '| patients:', DF.patient.nunique())
print(DF.groupby('fold').agg(rois=('roi_id','size'), patients=('patient','nunique')).T.to_string())


In [ ]:
# 3c · Disk-space fix: Keras stores the Adam optimizer state inside best.weights.h5 (~95 MB per run).
# After each run, the weights are re-saved WITHOUT the optimizer state (~31 MB) and the new file is
# verified weight-by-weight against the original before it replaces it. Model weights are unchanged.
def shrink_weights(run, root):
    d = f'{root}/{run}'; big = f'{d}/best.weights.h5'; small = f'{d}/best_small.weights.h5'
    if not os.path.exists(big) or os.path.exists(f'{d}/.shrunk'):
        return
    if os.path.getsize(big) < 60e6:
        open(f'{d}/.shrunk', 'w').close(); return
    name = run.split('_fold')[0]
    tf.keras.backend.clear_session()
    m1 = qseg2.build_p2_model(name, (cfg.image_size, cfg.image_size, cfg.in_channels), cfg.n_qubits, cfg.n_q_layers)
    m1.load_weights(big)
    m1.save_weights(small)
    m2 = qseg2.build_p2_model(name, (cfg.image_size, cfg.image_size, cfg.in_channels), cfg.n_qubits, cfg.n_q_layers)
    m2.load_weights(small)
    w1, w2 = m1.get_weights(), m2.get_weights()
    assert len(w1) == len(w2) and all(np.array_equal(a, b) for a, b in zip(w1, w2)), f'{run}: weight check failed, original kept'
    os.replace(small, big)
    open(f'{d}/.shrunk', 'w').close()
    print(f'   {run}: weights re-saved without optimizer state ({os.path.getsize(big) / 1e6:.0f} MB)', flush=True)


In [ ]:
# 4 · Part A: main schedule (ReduceLROnPlateau on val_loss, as in the paper). Fold 0, seed 0.
import time
DATA = (X, Y, DF)
cfg.to_json(f'{RUNS_DIR}/train_config.json')
def lr_check(root, run):
    h = pd.read_csv(f'{root}/{run}/history.csv')
    col = 'learning_rate' if 'learning_rate' in h else 'lr'
    low = h.index[h[col] <= 1e-6]
    return (int(low[0]) + 1) if len(low) else None, len(h)
def go(root, name):
    run = f'{name}_fold0_seed0'
    qseg2.run_fold(cfg, DATA, name, 0, 0, out_root=root, min_epochs=15, verbose=0)
    shrink_weights(run, root)
    ep_low, n_ep = lr_check(root, run)
    print(time.strftime('%H:%M UTC', time.gmtime()), f'| {root.split("/")[-1]}/{run}: {n_ep} epochs, lr<=1e-6 from epoch {ep_low}', flush=True)
for name in ['unet', 'g_q', 'g_cl', 'p_q', 'p_cl']:
    go(RUNS_DIR, name)
print('part A finished')


In [ ]:
# 5 · Part B: same, but the learning-rate schedule watches the TRAINING loss (set only after part A).
_Plateau = tf.keras.callbacks.ReduceLROnPlateau
if not getattr(_Plateau, '_p2_train_loss', False):
    class TrainLossPlateau(_Plateau):
        _p2_train_loss = True
        def __init__(self, monitor='val_loss', **kw):
            super().__init__(monitor='loss', **kw)
    tf.keras.callbacks.ReduceLROnPlateau = TrainLossPlateau
assert qseg2.tf.keras.callbacks.ReduceLROnPlateau(factor=0.5, patience=4).monitor == 'loss'
print('LR schedule now monitors:', qseg2.tf.keras.callbacks.ReduceLROnPlateau(factor=0.5, patience=4).monitor)

for name in ['p_q', 'p_cl']:
    go(LR_DIR, name)
print('part B finished')


In [ ]:
# 6 · Pilot results vs the no-network baselines of the same fold (pass rule in the header)
base = pd.read_csv(f'{PROJECT}/results/p2_jit_baselines.csv')
b0 = base[(base.fold == 0) & (base.dataset == 'CBIS')].set_index('baseline').dice
rows = []
for root, tag in ((RUNS_DIR, 'main'), (LR_DIR, 'lrtrain')):
    r = qseg2.collect(root)
    if len(r): r['schedule'] = tag; rows.append(r)
res = pd.concat(rows)
cols = ['schedule', 'model', 'dice_per_image', 'dice_sever_const', 'dice_sever_swap', 'best_epoch', 'epochs_run']
res = res[[c for c in cols if c in res]].copy()
res['minus_disc'] = (res['dice_per_image'] - b0['centred_disc']).round(3)
print('fold-0 baselines (CBIS):', b0.round(3).to_dict())
print(res.round(4).to_string(index=False))
res.to_csv(f'{PROJECT}/results/p2_jit_pilot_summary.csv', index=False)
need = res[(res.schedule == 'main') & res.model.isin(['unet', 'g_q', 'g_cl'])]
print('\nPASS' if (need.minus_disc >= 0.10).all() and len(need) == 3 else '\nNOT PASSED (see table)')


In [ ]:
# 7 · Auto-shutdown: save Drive, then release the GPU (runs only after cell 6 has printed the table)
import time
from google.colab import drive, runtime
print('training cell finished at', time.strftime('%H:%M:%S UTC', time.gmtime()))
drive.flush_and_unmount()
print('Drive flushed')
time.sleep(30)
runtime.unassign()
